In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import torch
from bgflow import MultiDoubleWellPotential
from bgflow.utils import distance_vectors, distances_from_vectors, remove_mean

In [ ]:
import os
import sys

# Get the absolute path to the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add the parent directory to sys.path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from src.utils.plots import distance_histogram, energy_histogram

# Now you can import modules from the source/ directory
from src.utils.tbg_utils import kish_effective_sample_size

## Define target

In [ ]:
# first define system dimensionality and a target energy/distribution

dim = 8
n_particles = 4
n_dimensions = dim // n_particles

# DW parameters
a = 0.9
b = -4
c = 0
offset = 4

target = MultiDoubleWellPotential(dim, n_particles, a, b, c, offset, two_event_dims=False)

## Load data

In [ ]:
# define a MCMC sampler to sample from the target energy

dw4_data = np.load("/Users/chatan/fast-tbg/data/dw4-dataidx.npy", allow_pickle=True)
all_data = remove_mean(dw4_data[0], n_particles, n_dimensions)
idx = dw4_data[1]
data = all_data[idx[:100000]]
val_data = all_data[idx[100000:500000]]
data_holdout = all_data[idx[-500000:]]

In [ ]:
# define paths
run = "2025-01-08_12-39-21"
DATA_PATH = f"/Users/chatan/fast-tbg/logs/eval/runs/{run}/npy_outputs"

# load samples
samples_proposal = torch.tensor(np.load(DATA_PATH + "/samples_proposal.npy"))
log_p_proposal = torch.tensor(np.load(DATA_PATH + "/log_p_proposal.npy"))

# destandardize samples
samples_proposal = samples_proposal.view(-1, 4, 2)
samples_proposal *= torch.tensor([1.8230, 1.8103])
samples_proposal = samples_proposal.view(-1, 8)

print(samples_proposal.shape)

## Compute importance weights

In [ ]:
# compute importance weights

logits = -target.energy(samples_proposal).flatten() + log_p_proposal.flatten()
max_logits = torch.max(logits)
importance_weights = torch.nn.functional.softmax(logits - max_logits)

## ESS

In [ ]:
print(kish_effective_sample_size(importance_weights) / len(importance_weights))

## Energy histogram

In [ ]:
energy_histogram(data_holdout, samples_proposal, importance_weights, target.energy)

## Distance histogram

In [ ]:
distance_fn = lambda x: distances_from_vectors(
    distance_vectors(x.view(-1, n_particles, n_dimensions))
).reshape(-1)
distance_histogram(
    data_holdout,
    samples_proposal,
    torch.repeat_interleave(importance_weights.flatten(), n_particles * (n_particles - 1)),
    distance_fn,
)